# 1 - import

In [19]:
import os
import json
import math
import random
from collections import Counter
from typing import List, Dict

import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

from sklearn.metrics import mean_squared_error

In [20]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

In [21]:
import csv
from datetime import datetime

# 2 - configs

In [22]:

CONFIG = {
    'annotations': r'D:\Balanced_20_Frames_Augmented\train_final.json',
    'data_root': r'D:\Balanced_20_Frames_Augmented\Train',
    'stats_file': r'D:\Balanced_20_Frames_Augmented\stats.json',
    'label_map': r'D:\Balanced_20_Frames_Augmented\label_map_final.json',  # optional
    'save_dir': './checkpoints_text2sign',
    'batch_size': 24,
    'epochs': 40,
    'lr': 1e-5,
    'patience': 5,
    'weight_decay': 1e-2,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'max_text_len': 32,
    'max_landmark_len': 70,
    'd_model': 512,
    'nhead': 8,
    'num_encoder_layers': 3,
    'num_decoder_layers': 3,
    'dropout': 0.1,
    'teacher_forcing_rate': 0.7,
    'save_every': 1,
    'seed': 42
}

os.makedirs(CONFIG['save_dir'], exist_ok=True)
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])

# 3 - tokenizer

In [23]:
# class SimpleTokenizer:
#     def __init__(self, texts: List[str], min_freq: int = 1, max_vocab: int = None):
#         # word-level tokenizer
#         counter = Counter()
#         for t in texts:
#             tokens = self._tok(t)
#             counter.update(tokens)
#         # special tokens
#         self.pad_token = '<pad>'
#         self.sos_token = '<sos>'
#         self.eos_token = '<eos>'
#         self.unk_token = '<unk>'
#         specials = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]
#         vocab_items = [w for w, c in counter.items() if c >= min_freq]
#         if max_vocab:
#             vocab_items = vocab_items[:max_vocab]
#         self.itos = specials + vocab_items
#         self.stoi = {w: i for i, w in enumerate(self.itos)}
#     def _tok(self, text: str):
#         # simple whitespace + lower
#         return text.lower().strip().split()
#     def encode(self, text: str, max_len: int):
#         toks = self._tok(text)
#         toks = toks[:max_len-2]  # reserve for sos/eos
#         ids = [self.stoi.get(t, self.stoi[self.unk_token]) for t in toks]
#         return [self.stoi[self.sos_token]] + ids + [self.stoi[self.eos_token]]
#     def decode(self, ids: List[int]):
#         words = []
#         for i in ids:
#             w = self.itos[i] if i < len(self.itos) else self.unk_token
#             if w in (self.sos_token, self.eos_token, self.pad_token):
#                 continue
#             words.append(w)
#         return " ".join(words)
#     @property
#     def vocab_size(self):
#         return len(self.itos)

In [24]:
class HFTokenizer:
    def __init__(self, texts, min_freq=1, max_vocab=None):
        # ---- Special tokens ----
        self.pad_token = "<pad>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"
        self.unk_token = "<unk>"
        specials = [self.pad_token, self.sos_token, self.eos_token, self.unk_token]

        # ---- Tokenizer backend ----
        self.tokenizer = Tokenizer(WordPiece(unk_token=self.unk_token))
        self.tokenizer.pre_tokenizer = Whitespace()

        if max_vocab is None:
            max_vocab = 5000

        trainer = WordPieceTrainer(
            special_tokens=specials,
            min_frequency=min_freq,
            vocab_size=max_vocab,  
        )

        # ---- Train on provided texts ----
        self.tokenizer.train_from_iterator(texts, trainer)

        # ---- stoi / itos ----
        self.stoi = self.tokenizer.get_vocab()
        self.itos = sorted(self.stoi, key=lambda w: self.stoi[w])

    @property
    def vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def encode(self, text, max_len):
        ids = self.tokenizer.encode(text.lower().strip()).ids
        ids = ids[:max_len - 2]
        return [self.stoi[self.sos_token]] + ids + [self.stoi[self.eos_token]]

    def decode(self, ids):
        out = []
        for i in ids:
            if i >= len(self.itos):
                continue
            tok = self.itos[i]
            if tok in (self.sos_token, self.eos_token, self.pad_token):
                continue
            out.append(tok)
        return " ".join(out)


# 4 - t2sDataset class

In [ ]:
class TextToSignDataset(Dataset):
    def __init__(self, annotations_path, data_root, stats_path, tokenizer,
                 max_text_len=32, max_landmark_len=70, min_frames=5):

        with open(annotations_path, 'r') as f:
            self.annotations = json.load(f)

        with open(stats_path, 'r') as f:
            stats = json.load(f)

        mean = np.array(stats["spatial_mean"] + stats["temporal_mean"], dtype=np.float32)
        std = np.array(stats["spatial_std"] + stats["temporal_std"], dtype=np.float32)
        std[std < 1e-6] = 1.0

        self.mean = mean
        self.std = std
        self.data_root = data_root
        self.tokenizer = tokenizer
        self.max_text_len = max_text_len
        self.max_landmark_len = max_landmark_len
        self.min_frames = min_frames

        self.samples = []
        print("[Dataset] Scanning annotations...")
        for entry in tqdm(self.annotations):
            gloss = entry.get("gloss", "").strip()
            if not gloss:
                continue
            for inst in entry.get("instances", []):
                vid = inst.get("video_id")
                if not vid:
                    continue
                p = os.path.join(self.data_root, vid, "landmarks.json")
                if os.path.exists(p):
                    self.samples.append({"video_id": vid, "landmarks_path": p, "text": gloss})

        print(f"[Dataset] Total usable pairs: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def _extract_spatial_features(self, frame):
        if not isinstance(frame, dict):
            return None

        def safe_get(key, size):
            x = np.array(frame.get(key, []), dtype=np.float32).flatten()
            if len(x) > size:
                return x[:size]
            if len(x) < size:
                return np.pad(x, (0, size - len(x)))
            return x

        try:
            return np.concatenate([
                safe_get("pose", 132),
                safe_get("left_hand", 84),
                safe_get("right_hand", 84),
                safe_get("face", 1404),
                safe_get("left_hand_engineered", 19),
                safe_get("right_hand_engineered", 19),
            ])
        except:
            return None

    def __getitem__(self, idx):
        sample = self.samples[idx]
        path = sample["landmarks_path"]
        text = sample["text"]

        try:
            with open(path, "r") as f:
                frames = json.load(f)
            if not isinstance(frames, list) or len(frames) < self.min_frames:
                return None
        except:
            return None

        feats = [self._extract_spatial_features(fr) for fr in frames]
        if any(f is None for f in feats):
            return None

        spatial = np.array(feats, dtype=np.float32)
        if np.isnan(spatial).any() or np.isinf(spatial).any():
            return None

        temporal = np.diff(spatial, axis=0, prepend=spatial[:1])
        features = np.concatenate([spatial, temporal], axis=1)

        if features.shape[1] != self.mean.shape[0]:
            return None

        features = (features - self.mean) / self.std
        
        features = np.nan_to_num(
            features,
            nan=0.0,
            posinf=5.0,
            neginf=-5.0
        )
        features = np.clip(features, -10.0, 10.0)


        T, D = features.shape
        if T >= self.max_landmark_len:
            idxs = np.linspace(0, T - 1, self.max_landmark_len, dtype=int)
            features = features[idxs]
            lmask = np.ones(self.max_landmark_len, dtype=np.float32)
        else:
            pad = np.zeros((self.max_landmark_len - T, D), dtype=np.float32)
            features = np.concatenate([features, pad], axis=0)
            lmask = np.array([1]*T + [0]*(self.max_landmark_len - T), dtype=np.float32)

        tokens = self.tokenizer.encode(text, max_len=self.max_text_len)
        if len(tokens) < self.max_text_len:
            tokens += [self.tokenizer.stoi[self.tokenizer.pad_token]] * (self.max_text_len - len(tokens))
        else:
            tokens = tokens[:self.max_text_len]

        imask = [1 if t != self.tokenizer.stoi[self.tokenizer.pad_token] else 0 for t in tokens]

        return {
            "input_ids": torch.tensor(tokens, dtype=torch.long),
            "input_mask": torch.tensor(imask, dtype=torch.bool),
            "landmarks": torch.tensor(features, dtype=torch.float32),
            "landmark_mask": torch.tensor(lmask, dtype=torch.bool),
        }


def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return None

    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "input_mask": torch.stack([b["input_mask"] for b in batch]),
        "landmarks": torch.stack([b["landmarks"] for b in batch]),
        "landmark_mask": torch.stack([b["landmark_mask"] for b in batch]),
    }


# 5 - t2s model

In [ ]:
def generate_square_subsequent_mask(sz: int, device):
    mask = torch.triu(torch.ones((sz, sz), device=device) * float('-inf'), diagonal=1)
    return mask  # shape (sz, sz)

class PositionalEmbedding(nn.Module):
    def __init__(self, max_len: int, d_model: int):
        super().__init__()
        self.pos_emb = nn.Embedding(max_len, d_model)
    def forward(self, seq_len):
        # returns (seq_len, d_model)
        positions = torch.arange(seq_len, device=self.pos_emb.weight.device)
        return self.pos_emb(positions)  # (seq_len, d_model)

class TextToSignModel(nn.Module):
    def __init__(self, vocab_size, landmark_dim, cfg: Dict):
        super().__init__()
        d_model = cfg['d_model']
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.landmark_dim = landmark_dim
        self.max_landmark_len = cfg['max_landmark_len']
        self.max_text_len = cfg['max_text_len']

        # text encoder
        # print("[Model] Initializing text encoder...")
        self.tok_embed = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.text_pos = PositionalEmbedding(self.max_text_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=cfg['nhead'],
                                                   dim_feedforward=d_model*4, dropout=cfg['dropout'],
                                                   batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=cfg['num_encoder_layers'])

        # print("[Model] Building landmark projection...")
        self.landmark_in_proj = nn.Linear(landmark_dim, d_model)
        self.start_frame = nn.Parameter(torch.randn(1, d_model))  # (1, d_model)
        self.landmark_pos = PositionalEmbedding(self.max_landmark_len, d_model)

        # transformer decoder
        # print("[Model] Initializing decoder...")
        decoder_layer = nn.TransformerDecoderLayer(d_model=d_model, nhead=cfg['nhead'],
                                                   dim_feedforward=d_model*4, dropout=cfg['dropout'],
                                                   batch_first=True, activation='gelu')
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=cfg['num_decoder_layers'])

        # output projection
        self.out_proj = nn.Linear(d_model, landmark_dim)

        # print("[Model] Applying weight initialization...")
        self._init_weights()
        # print("[Model] Initialization complete.\n")

    def _init_weights(self):
        for n, p in self.named_parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode_text(self, input_ids, input_mask):
        # print("[Model] Encoding text...")
        emb = self.tok_embed(input_ids)  # (B, T_text, d_model)
        pos = self.text_pos(emb.shape[1]).unsqueeze(0)  # (1, T_text, d_model)
        src = emb + pos  # (B, T_text, d_model)
        src_key_padding_mask = ~input_mask  # True = padding
        memory = self.encoder(src, src_key_padding_mask=src_key_padding_mask)  # (B, T_text, d_model)
        # print("[Model] Text encoding complete.")
        return memory, src_key_padding_mask

    def forward(self, input_ids, input_mask, tgt_landmarks=None, teacher_forcing_ratio=0.9):
        """
        If tgt_landmarks provided -> training with teacher forcing.
        - input_ids: (B, T_text)
        - input_mask: (B, T_text) bool
        - tgt_landmarks: (B, T_landmark, landmark_dim) (normalized)
        Returns:
            outputs: (B, T_landmark, landmark_dim)
        """

        # print("[Model] Starting forward pass...")
        B = input_ids.shape[0]
        device = input_ids.device
        memory, src_key_padding_mask = self.encode_text(input_ids, input_mask)

        max_T = self.max_landmark_len
        d = self.d_model

        start = self.start_frame.unsqueeze(0).expand(B, -1, -1)  # (B, 1, d_model)
        outputs = []
        pos_all = self.landmark_pos(max_T).unsqueeze(0)  # (1, T_landmark, d_model)

        prev = start  # (B, 1, d_model)

        # print(f"[Model] Decoding {max_T} frames...")
        for t in tqdm(range(max_T), desc="Decoding", leave=False):
            tgt = torch.cat(outputs + [prev], dim=1) if outputs else prev  # (B, cur_len, d_model)
            cur_len = tgt.shape[1]
            tgt = tgt + pos_all[:, :cur_len, :]
            tgt_mask = generate_square_subsequent_mask(cur_len, device=device)  # (cur_len, cur_len)

            dec_out = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_key_padding_mask=src_key_padding_mask)
            last = dec_out[:, -1:, :]
            pred_frame = self.out_proj(last)

            outputs.append(self.landmark_in_proj(pred_frame).detach() * 0 + last)

            if (tgt_landmarks is not None) and (random.random() < teacher_forcing_ratio):
                gt_frame = tgt_landmarks[:, t:t+1, :]
                prev = self.landmark_in_proj(gt_frame)
            else:
                prev = self.landmark_in_proj(pred_frame)

        dec_feats = torch.cat(outputs, dim=1)
        preds = self.out_proj(dec_feats)

        # print("[Model] Forward pass complete.\n")
        return preds


# 6 - training and eval

In [27]:
def landmark_accuracy(y_pred, y_true, mask, alpha=5.0):
    """
    y_pred, y_true: [frames, dim]
    mask: [frames] boolean mask for valid frames
    """

    valid = mask.bool()
    if valid.sum() == 0:
        return None

    y_pred = y_pred[valid]
    y_true = y_true[valid]

    # reshape to [frames, keypoints, 2]
    F, D = y_pred.shape
    K = D // 2
    y_pred = y_pred.reshape(F, K, 2)
    y_true = y_true.reshape(F, K, 2)

    dist = torch.norm(y_pred - y_true, dim=-1)  # [frames, keypoints]
    mean_dist = dist.mean()   # scalar

    acc = torch.exp(-mean_dist / alpha)
    return acc.item()


In [28]:
class MaskedMSELoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, preds, targets, mask):
        # mask: (B, T) or (B, T, 1)
        if mask.dim() == preds.dim() - 1:
            mask = mask.unsqueeze(-1)

        mask = mask.float()

        diff = (preds - targets) ** 2
        diff = diff * mask

        denom = mask.sum().clamp(min=1.0)
        return diff.sum() / denom


In [29]:
def train_one_epoch(model, dataloader, optimizer, criterion, cfg, device):
    print("\n[Train] Starting epoch...")
    model.train()
    total_loss = 0.0
    count = 0

    for batch in tqdm(dataloader, desc="Train", leave=False):
        if batch is None:
            continue

        input_ids = batch['input_ids'].to(device)
        input_mask = batch['input_mask'].to(device)
        landmarks = batch['landmarks'].to(device)
        landmark_mask = batch['landmark_mask'].to(device)

        optimizer.zero_grad()

        preds = model(
            input_ids,
            input_mask,
            tgt_landmarks=landmarks,
            teacher_forcing_ratio=cfg['teacher_forcing_rate']
        )

        loss = criterion(preds, landmarks, landmark_mask)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        count += 1

    avg_loss = total_loss / max(1, count)
    print(f"[Train] Epoch complete. Avg Loss: {avg_loss:.6f}")
    return avg_loss


import csv
import os

@torch.no_grad()
def validate(model, dataloader, criterion, cfg, device, epoch=None):
    print("\n[Val] Running validation...")
    model.eval()
    
    total_loss = 0.0
    count = 0

    all_mse = []
    all_acc = []

    csv_path = os.path.join(cfg['save_dir'], "val_sample_metrics.csv")

    write_header = not os.path.exists(csv_path)
    csv_file = open(csv_path, mode='a', newline='')
    writer = csv.writer(csv_file)
    if write_header:
        writer.writerow(["epoch", "sample_index", "mse", "accuracy"])

    global_sample_index = 0 

    for batch in tqdm(dataloader, desc="Val", leave=False):
        if batch is None:
            continue

        input_ids = batch['input_ids'].to(device)
        input_mask = batch['input_mask'].to(device)
        landmarks = batch['landmarks'].to(device)
        landmark_mask = batch['landmark_mask'].to(device)

        preds = model(
            input_ids,
            input_mask,
            tgt_landmarks=None,
            teacher_forcing_ratio=0.0
        )

        loss = criterion(preds, landmarks, landmark_mask)
        total_loss += loss.item()
        count += 1

        bs = preds.shape[0]

        # Per sample metrics
        for i in range(bs):
            mask = landmark_mask[i]
            if mask.sum() == 0:
                global_sample_index += 1
                continue

            y_true = landmarks[i]
            y_pred = preds[i]

            # MSE
            m = mask.cpu().numpy().astype(bool)
            y_t = y_true.cpu().numpy()[m].reshape(-1)
            y_p = y_pred.cpu().numpy()[m].reshape(-1)
            mse = mean_squared_error(y_t, y_p)
            all_mse.append(mse)

            # Accuracy metric
            acc = landmark_accuracy(
                y_pred=y_pred,
                y_true=y_true,
                mask=mask
            )
            if acc is not None:
                all_acc.append(acc)

            writer.writerow([
                epoch if epoch is not None else -1,
                global_sample_index,
                mse,
                acc
            ])

            global_sample_index += 1

    csv_file.close()  

    avg_loss = total_loss / max(1, count)
    avg_mse = np.mean(all_mse) if len(all_mse) > 0 else None
    avg_acc = np.mean(all_acc) if len(all_acc) > 0 else None

    print(f"[Val] Loss: {avg_loss:.6f}, MSE: {avg_mse:.4f}, Acc: {avg_acc:.4f}")
    return avg_loss, avg_mse, avg_acc


In [30]:
@torch.no_grad()
def generate_from_text(model, tokenizer: HFTokenizer, text: str, cfg, device):
    model.eval()
    enc = tokenizer.encode(text, max_len=cfg['max_text_len'])
    if len(enc) < cfg['max_text_len']:
        enc = enc + [tokenizer.stoi[tokenizer.pad_token]] * (cfg['max_text_len'] - len(enc))
    input_ids = torch.tensor([enc], dtype=torch.long).to(device)
    input_mask = torch.tensor([[1 if i != tokenizer.stoi[tokenizer.pad_token] else 0 for i in enc]], dtype=torch.bool).to(device)
    preds = model(input_ids, input_mask, tgt_landmarks=None, teacher_forcing_ratio=0.0)  # (1, T, D)
    preds = preds.cpu().numpy()[0]  # (T, D)
    return preds


# 7 - main

In [31]:
cfg_init = CONFIG

In [32]:
log_path = os.path.join(cfg_init['save_dir'], "training_log.csv")

log_path = os.path.join(cfg_init['save_dir'], "training_log.csv")

if not os.path.exists(log_path):
    with open(log_path, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "val_mse", "val_acc", "timestamp"])


In [33]:

def main(cfg):
    device = cfg['device']
    print(f"[Run] device: {device}")

    # 1) tokenization of glosses
    with open(cfg['annotations'], 'r') as f:
        ann = json.load(f)
    texts = [e.get('gloss', '').strip() for e in ann if e.get('gloss', '').strip()]
    tokenizer = HFTokenizer(texts, min_freq=1)
    print(f"[Vocab] size: {tokenizer.vocab_size}")

    # 2) build dataset
    dataset = TextToSignDataset(cfg['annotations'], cfg['data_root'], cfg['stats_file'],
                                tokenizer, max_text_len=cfg['max_text_len'],
                                max_landmark_len=cfg['max_landmark_len'])
    # quick split
    val_frac = 0.15
    n_val = max(1, int(len(dataset)*val_frac))
    n_train = len(dataset) - n_val
    train_ds, val_ds = random_split(dataset, [n_train, n_val])
    print(f"[Split] train={len(train_ds)}, val={len(val_ds)}")

    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              collate_fn=collate_fn, num_workers=0, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=cfg['batch_size'], shuffle=False,
                            collate_fn=collate_fn, num_workers=0)

    # determine landmark dim by sampling one item
    sample_item = None
    for i in range(len(dataset)):
        s = dataset[i]
        if s is not None:
            sample_item = s; break
    if sample_item is None:
        raise RuntimeError("No valid samples found.")
    landmark_dim = sample_item['landmarks'].shape[1]
    print(f"[Landmark dim] {landmark_dim}")

    # 3) model
    model = TextToSignModel(vocab_size=tokenizer.vocab_size, landmark_dim=landmark_dim, cfg=cfg).to(device)
    print(f"[Model] params: {sum(p.numel() for p in model.parameters()):,}")

    # 4) optimizer / scheduler / criterion
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = MaskedMSELoss()
    patience = 5
    epochs_no_improve = 0

    best_val_loss = float('inf')
    for epoch in range(cfg['epochs']):
        print(f"\n=== Epoch {epoch+1}/{cfg['epochs']} ===")
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, cfg, device)
        val_loss, val_mse, val_acc = validate(model, val_loader, criterion, cfg, device)
        print(f"Train loss: {train_loss:.6f} | Val loss: {val_loss:.6f} | Val MSE (per sample): {val_mse} | Val Acc : {val_acc}")

        with open(log_path, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch+1,
                train_loss,
                val_loss,
                val_mse,
                val_acc,
                datetime.now().isoformat()
            ])

        
        # save checkpoint
        if (epoch+1) % cfg['save_every'] == 0:
            path = os.path.join(cfg['save_dir'], f'text2sign_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'tokenizer': tokenizer.itos,
                'cfg': cfg
            }, path)
            print(f"[Saved] {path}")

        # early stop
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({
                'epoch': epoch+1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'tokenizer': tokenizer.itos,
                'cfg': cfg
            }, os.path.join(cfg['save_dir'], 'best_text2sign.pth'))
            print("[Saved] best_text2sign.pth")
        else:
            epochs_no_improve += 1
            print(f"[Early Stop] No improvement for {epochs_no_improve}/{patience} epochs.")
            
        if epochs_no_improve == patience:
            print(f"== Early stopping triggered after {epoch+1} epochs (patience={patience}). ==")
            break

    # Example generation on first few validation samples
    print("\n[Generation examples]")
    model.eval()
    for i in range(min(5, len(val_ds))):
        sample = val_ds[i]
        if sample is None: continue
        text = tokenizer.decode(sample['input_ids'].numpy().tolist())
        preds = generate_from_text(model, tokenizer, text, cfg, device)
        # Un-normalize preds back to original space
        with open(cfg['stats_file'], 'r') as f:
            stats = json.load(f)
        mean = np.array(stats.get('spatial_mean', []) + stats.get('temporal_mean', []), dtype=np.float32)
        std = np.array(stats.get('spatial_std', []) + stats.get('temporal_std', []), dtype=np.float32)
        std[std < 1e-6] = 1.0
        preds_unnorm = preds * std + mean
        print(f"Text: {text}")
        print(f"Generated frames shape: {preds_unnorm.shape}")
        # Save prediction array
        save_arr = preds_unnorm.astype(np.float32)
        np.save(os.path.join(cfg['save_dir'], f"pred_{i}.npy"), save_arr)

        # Also save CSV
        np.savetxt(
            os.path.join(cfg['save_dir'], f"pred_{i}.csv"),
            save_arr.reshape(save_arr.shape[0], -1),
            delimiter=","
        )

        print(f"Saved pred_{i}.npy and pred_{i}.csv")
        print(f"Text: {text} | Frames: {preds_unnorm.shape[0]}")



In [ ]:
main(CONFIG)

[Run] device: cuda
[Vocab] size: 572
[Dataset] Scanning annotations...


100%|██████████| 200/200 [00:00<00:00, 1451.09it/s]


[Dataset] Total usable pairs: 10000
[Split] train=8500, val=1500
[Landmark dim] 3484
[Model] params: 25,986,460


c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== Epoch 1/40 ===

[Train] Starting epoch...


...

[Train] Epoch complete. Avg Loss: 2207.209801

[Val] Running validation...


[Val] Loss: 3307.199728, MSE: 0.8945, Acc: 0.8148
Train loss: 2207.209801 | Val loss: 3307.199728 | Val MSE (per sample): 0.8944923289897252 | Val Acc : 0.8147977393097673
[Saved] ./checkpoints_text2sign\text2sign_epoch_9.pth
[Saved] best_text2sign.pth

=== Epoch 10/40 ===

[Train] Starting epoch...


[Train] Epoch complete. Avg Loss: 2230.424574

[Val] Running validation...


[Val] Loss: 3365.209291, MSE: 0.9202, Acc: 0.8119
Train loss: 2230.424574 | Val loss: 3365.209291 | Val MSE (per sample): 0.9202344333467308 | Val Acc : 0.8118873253190444
[Saved] ./checkpoints_text2sign\text2sign_epoch_10.pth
[Early Stop] No improvement for 1/5 epochs.

=== Epoch 11/40 ===

[Train] Starting epoch...


Train:  21%|██▏       | 76/355 [03:39<11:18,  2.43s/it]Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001F323F0E120>>
Traceback (most recent call last):
  File "c:\Users\tahmi\Documents\Work\Text2Sign\t2slt\train_env\Lib\site-packages\ipykernel\ipkernel.py", line 794, in _clean_thread_parent_frames
    for identity in list(thread_to_parent_header.keys()):
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 
Train:  29%|██▊       | 102/355 [04:40<09:54,  2.35s/it]

# 8 - Testing

In [54]:
import torch, json, os, numpy as np

cfg = CONFIG

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg['device'] = device

print(f"[Run] device: {device}")

ckpt_path = "checkpoints_text2sign/best_text2sign_try1.pth"
checkpoint = torch.load(ckpt_path, map_location=device)
expected_vocab = checkpoint['model_state_dict']['tok_embed.weight'].shape[0]
print(f"[Checkpoint vocab] = {expected_vocab}")

with open(cfg['annotations'], 'r') as f:
    ann = json.load(f)
texts = [e.get('gloss', '').strip() for e in ann if e.get('gloss', '').strip()]

tokenizer = HFTokenizer(texts, min_freq=1)

if tokenizer.vocab_size != expected_vocab:
    print(f"[Tokenizer mismatch] new={tokenizer.vocab_size}, expected={expected_vocab}")

    # ---- Trim stoi/itos to expected vocab size ----
    trimmed_itos = tokenizer.itos[:expected_vocab]
    trimmed_stoi = {tok: idx for idx, tok in enumerate(trimmed_itos)}

    # ---- Replace tokenizer model with trimmed vocab ----
    from tokenizers.models import WordPiece
    tokenizer.tokenizer.model = WordPiece(vocab=trimmed_stoi, unk_token=tokenizer.unk_token)

    # ---- Assign back ----
    tokenizer.itos = trimmed_itos
    tokenizer.stoi = trimmed_stoi

    assert tokenizer.vocab_size == expected_vocab, "Tokenizer still mismatched!"
    print("== > Tokenizer vocab trimmed to checkpoint vocab")


print(f"[Tokenizer final vocab] = {tokenizer.vocab_size}")

dataset = TextToSignDataset(cfg['annotations'], cfg['data_root'], cfg['stats_file'],
                            tokenizer, max_text_len=cfg['max_text_len'],
                            max_landmark_len=cfg['max_landmark_len'])

val_frac = 0.15
n_val = max(1, int(len(dataset)*val_frac))
n_train = len(dataset) - n_val
train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val])

for s in dataset:
    if s is not None:
        landmark_dim = s['landmarks'].shape[1]
        break

print(f"[Landmark dim] {landmark_dim}")

model = TextToSignModel(tokenizer.vocab_size, landmark_dim, cfg).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("==>  Model loaded")

os.makedirs(cfg['save_dir'], exist_ok=True)

with torch.no_grad():
    for i in range(min(5, len(val_ds))):
        sample = val_ds[i]
        if sample is None: continue

        text = tokenizer.decode(sample['input_ids'].numpy().tolist())
        preds = generate_from_text(model, tokenizer, text, cfg, device)

        stats = json.load(open(cfg['stats_file']))
        mean = np.array(stats['spatial_mean']+stats['temporal_mean'],dtype=np.float32)
        std = np.array(stats['spatial_std']+stats['temporal_std'],dtype=np.float32)
        std[std<1e-6]=1

        preds_unnorm = preds*std + mean
        save_arr = preds_unnorm.astype(np.float32)

        np.save(os.path.join(cfg['save_dir'],f"pred_{i}.npy"), save_arr)
        np.savetxt(os.path.join(cfg['save_dir'],f"pred_{i}.csv"),
                   save_arr.reshape(save_arr.shape[0],-1), delimiter=",")

        print(f"[OK] Text: {text} → {save_arr.shape} frames saved")


[Run] device: cuda
[Checkpoint vocab] = 572
[Tokenizer mismatch] new=577, expected=572
== > Tokenizer vocab trimmed to checkpoint vocab
[Tokenizer final vocab] = 572
[Dataset] Scanning annotations...


100%|██████████| 200/200 [00:00<00:00, 1817.15it/s]

[Dataset] Total usable pairs: 10000


[Landmark dim] 3484
==>  Model loaded


[OK] Text: delay → (70, 3484) frames saved


In [55]:
import numpy as np
import cv2
import mediapipe as mp

INPUT_FILE = cfg['save_dir'] + "/pred_4.npy"
OUTPUT_VIDEO = "pred_delay_output.mp4"
FPS = 15
IMG_SIZE = 800  

POSE = 33
FACE = 468
HAND = 21

POSE_DIM = POSE * 4
FACE_DIM = FACE * 3
HAND_DIM = HAND * 3

SPATIAL_TOTAL = POSE_DIM + FACE_DIM + HAND_DIM*2

mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face = mp.solutions.face_mesh

def split_frame(vec):
    spatial = vec[:SPATIAL_TOTAL]

    pose = spatial[:POSE_DIM].reshape(POSE,4)
    i = POSE_DIM

    face = spatial[i:i+FACE_DIM].reshape(FACE,3)
    i += FACE_DIM

    left = spatial[i:i+HAND_DIM].reshape(HAND,3)
    i += HAND_DIM

    right = spatial[i:i+HAND_DIM].reshape(HAND,3)

    # XY only 
    return pose[:, :2], face[:, :2], left[:, :2], right[:, :2]

data = np.load(INPUT_FILE)
frames = data.shape[0]

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, FPS, (IMG_SIZE, IMG_SIZE))

POSE_CONN = mp_pose.POSE_CONNECTIONS
HAND_CONN = mp_hands.HAND_CONNECTIONS
FACE_CONN = mp_face.FACEMESH_TESSELATION  # (or FACEMESH_CONTOURS for cleaner)

def norm(xy):
    x = (xy[:,0] * IMG_SIZE).astype(int)
    y = (xy[:,1] * IMG_SIZE).astype(int)
    return np.stack([x,y], axis=1)

print("Rendering video...")

for t in range(frames):
    frame = np.ones((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) * 255

    pose, face, left, right = split_frame(data[t])

    pose = norm(pose); face = norm(face); left = norm(left); right = norm(right)

    for a,b in POSE_CONN:
        cv2.line(frame, tuple(pose[a]), tuple(pose[b]), (0,0,255), 2)

    for a,b in HAND_CONN:
        cv2.line(frame, tuple(left[a]), tuple(left[b]), (255,0,0), 2)
        cv2.line(frame, tuple(right[a]), tuple(right[b]), (0,255,0), 2)

    for a,b in FACE_CONN:
        cv2.line(frame, tuple(face[a]), tuple(face[b]), (200,200,200), 1)

    for p in pose: cv2.circle(frame, tuple(p), 4, (0,0,200), -1)
    for p in left: cv2.circle(frame, tuple(p), 4, (200,0,0), -1)
    for p in right: cv2.circle(frame, tuple(p), 4, (0,200,0), -1)

    writer.write(frame)

writer.release()
print(f"== Video saved to {OUTPUT_VIDEO}")


Rendering video...
== Video saved to pred_delay_output.mp4
